In [2]:
# ==============================================================================
# CELL 1: Environment & Dataset Parser (Zero External Pip Dependencies for Metrics)
# ==============================================================================
import os
import glob
import re
import gc
import warnings
import dataclasses
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Execution Device: {device}")

# Native Python Levenshtein Distance for WER and CER (No 'evaluate' library needed)
def levenshtein_distance(ref_tokens: List[str], hyp_tokens: List[str]) -> int:
    d = np.zeros((len(ref_tokens) + 1, len(hyp_tokens) + 1), dtype=np.int32)
    for i in range(len(ref_tokens) + 1):
        d[i, 0] = i
    for j in range(len(hyp_tokens) + 1):
        d[0, j] = j
    for i in range(1, len(ref_tokens) + 1):
        for j in range(1, len(hyp_tokens) + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                d[i, j] = d[i - 1, j - 1]
            else:
                d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + 1)
    return d[len(ref_tokens), len(hyp_tokens)]

def compute_wer(predictions: List[str], references: List[str]) -> float:
    total_words, total_errs = 0, 0
    for p, r in zip(predictions, references):
        r_words = r.strip().split()
        p_words = p.strip().split()
        total_words += len(r_words)
        total_errs += levenshtein_distance(r_words, p_words)
    return (total_errs / max(1, total_words)) * 100.0

def compute_cer(predictions: List[str], references: List[str]) -> float:
    total_chars, total_errs = 0, 0
    for p, r in zip(predictions, references):
        r_chars = list(r.replace(" ", ""))
        p_chars = list(p.replace(" ", ""))
        total_chars += len(r_chars)
        total_errs += levenshtein_distance(r_chars, p_chars)
    return (total_errs / max(1, total_chars)) * 100.0

# District and Division Name Mappings
DISTRICT_TO_BANGLA_NAME = {
    "barisal": "বরিশাল", "bhola": "ভোলা", "jhalokati": "ঝালকাঠি", "patuakhali": "পটুয়াখালী",
    "barguna": "বরগুনা", "pirojpur": "পিরোজপুর", "chittagong": "চট্টগ্রাম", "comilla": "কুমিল্লা",
    "feni": "ফেনী", "brahmanbaria": "ব্রাহ্মণবাড়িয়া", "noakhali": "নোয়াখালী", "laksmipur": "লক্ষ্মীপুর",
    "lakshmipur": "লক্ষ্মীপুর", "chandpur": "চাঁদপুর", "coxsbazar": "কক্সবাজার", "dhaka": "ঢাকা",
    "faridpur": "ফরিদপুর", "gazipur": "গাজীপুর", "kishoreganj": "কিশোরগঞ্জ", "manikganj": "মানিকগঞ্জ",
    "munshiganj": "মুন্সিগঞ্জ", "narayanganj": "নারায়ণগঞ্জ", "narsingdi": "নরসিংদী", "rajbari": "রাজবাড়ী",
    "shariatpur": "শরীয়তপুর", "tangail": "টাঙ্গাইল", "mymensingh": "ময়মনসিংহ", "netrokona": "নেত্রকোণা",
    "jamalpur": "জামালপুর", "sherpur": "শেরপুর", "khulna": "খুলনা", "jessore": "যশোর",
    "kushtia": "কুষ্টিয়া", "satkhira": "সাতক্ষীরা", "bagerhat": "বাগেরহাট", "chuadanga": "চুয়াডাঙ্গা",
    "jhenaidah": "ঝিনাইদহ", "magura": "মাগুরা", "meherpur": "মেহেরপুর", "narail": "নড়াইল",
    "rajshahi": "রাজশাহী", "bogura": "বগুড়া", "bogra": "বগুড়া", "chapainawabganj": "চাঁপাইনবাবগঞ্জ",
    "joypurhat": "জয়পুরহাট", "naogaon": "নওগাঁ", "natore": "নাটোর", "pabna": "পাবনা",
    "sirajganj": "সিরাজগঞ্জ", "rangpur": "রংপুর", "dinajpur": "দিনাজপুর", "gaibandha": "গাইবান্ধা",
    "kurigram": "কুড়িগ্রাম", "lalmonirhat": "লালমনিরহাট", "nilphamari": "নীলফামারী",
    "panchagarh": "পঞ্চগড়", "thakurgaon": "ঠাকুরগাঁও", "sylhet": "সিলেট", "habiganj": "হবিগঞ্জ",
    "moulvibazar": "মৌলভীবাজার", "sunamganj": "সুনামগঞ্জ"
}

DISTRICT_TO_BANGLA_DIV = {
    "barisal": "বরিশাল", "bhola": "বরিশাল", "jhalokati": "বরিশাল", "patuakhali": "বরিশাল",
    "barguna": "বরিশাল", "pirojpur": "বরিশাল", "chittagong": "চট্টগ্রাম", "comilla": "চট্টগ্রাম",
    "feni": "চট্টগ্রাম", "brahmanbaria": "চট্টগ্রাম", "noakhali": "চট্টগ্রাম", "laksmipur": "চট্টগ্রাম",
    "lakshmipur": "চট্টগ্রাম", "chandpur": "চট্টগ্রাম", "coxsbazar": "চট্টগ্রাম", "dhaka": "ঢাকা",
    "faridpur": "ঢাকা", "gazipur": "ঢাকা", "kishoreganj": "ঢাকা", "manikganj": "ঢাকা",
    "munshiganj": "ঢাকা", "narayanganj": "ঢাকা", "narsingdi": "ঢাকা", "rajbari": "ঢাকা",
    "shariatpur": "ঢাকা", "tangail": "ঢাকা", "mymensingh": "ময়মনসিংহ", "netrokona": "ময়মনসিংহ",
    "jamalpur": "ময়মনসিংহ", "sherpur": "ময়মনসিংহ", "khulna": "খুলনা", "jessore": "খুলনা",
    "kushtia": "খুলনা", "satkhira": "খুলনা", "bagerhat": "খুলনা", "chuadanga": "খুলনা",
    "jhenaidah": "খুলনা", "magura": "খুলনা", "meherpur": "খুলনা", "narail": "খুলনা",
    "rajshahi": "রাজশাহী", "bogura": "রাজশাহী", "bogra": "রাজশাহী", "chapainawabganj": "রাজশাহী",
    "joypurhat": "রাজশাহী", "naogaon": "রাজশাহী", "natore": "রাজশাহী", "pabna": "রাজশাহী",
    "sirajganj": "রাজশাহী", "rangpur": "রংপুর", "dinajpur": "রংপুর", "gaibandha": "রংপুর",
    "kurigram": "রংপুর", "lalmonirhat": "রংপুর", "nilphamari": "রংপুর", "panchagarh": "রংপুর",
    "thakurgaon": "রংপুর", "sylhet": "সিলেট", "habiganj": "সিলেট", "moulvibazar": "সিলেট",
    "sunamganj": "সিলেট"
}

def district_token_from_filename(filepath: str) -> str:
    stem = os.path.splitext(os.path.basename(filepath))[0].lower()
    parts = stem.split("_")
    if parts and parts[0] in {"male", "female", "m", "f"}:
        parts = parts[1:]
    if parts and parts[-1].isdigit():
        parts = parts[:-1]
    token = "_".join(parts)
    if token in DISTRICT_TO_BANGLA_NAME:
        return token
    for key in DISTRICT_TO_BANGLA_NAME:
        if key in token:
            return key
    return token

# Dataset Parsing
dataset_dir = "/kaggle/input/datasets/prosenjitmondol/bangla-regional/shobdotori"
if not os.path.exists(dataset_dir):
    dataset_dir = "/kaggle/input/bangla-regional/shobdotori"

train_dir = os.path.join(dataset_dir, "Train")
annotation_dir = os.path.join(dataset_dir, "Train_annotation")

csv_files = glob.glob(os.path.join(annotation_dir, "*.csv"))
dfs = []
for csv_path in csv_files:
    folder_name = os.path.splitext(os.path.basename(csv_path))[0]
    audio_dir = os.path.join(train_dir, folder_name)
    sub = pd.read_csv(csv_path)
    sub.columns = [c.strip() for c in sub.columns]
    audio_col, text_col = sub.columns[0], sub.columns[1]
    div_col = sub.columns[2] if len(sub.columns) >= 3 else None
    
    keep = [audio_col, text_col] + ([div_col] if div_col else [])
    sub = sub[keep].copy()
    sub.columns = ["audio_id", "sentence"] + (["division_en"] if div_col else [])
    if not div_col:
        sub["division_en"] = folder_name

    def make_path(val, base=audio_dir):
        fn = str(val).strip()
        if not fn.lower().endswith(".wav"):
            fn += ".wav"
        return os.path.join(base, fn)

    sub["audio_path"] = sub["audio_id"].apply(make_path)
    sub["district_token"] = sub["audio_id"].apply(district_token_from_filename)
    sub["জেলা"] = sub["district_token"].apply(lambda t: DISTRICT_TO_BANGLA_NAME.get(t, t))
    sub["বিভাগ"] = sub["district_token"].apply(lambda t: DISTRICT_TO_BANGLA_DIV.get(t, "অন্যান্য"))
    dfs.append(sub)

full_df = pd.concat(dfs, ignore_index=True).dropna(subset=["sentence"])
full_df = full_df[full_df["audio_path"].apply(os.path.exists)].reset_index(drop=True)

train_df, eval_df = train_test_split(full_df, test_size=0.1, random_state=42)
train_df, eval_df = train_df.reset_index(drop=True), eval_df.reset_index(drop=True)
print(f"[✓] Data Loaded → Total Valid Samples: {len(full_df)} | Train: {len(train_df)} | Validation: {len(eval_df)}")

[*] Execution Device: cuda
[✓] Data Loaded → Total Valid Samples: 3350 | Train: 3015 | Validation: 335


In [ ]:
# ==============================================================================
# CELL 2: Model 1 — Whisper-Small Fine-Tuning
# ==============================================================================
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

whisper_base_model = "bangla-speech-processing/BanglaASR"
whisper_processor = WhisperProcessor.from_pretrained(whisper_base_model)
whisper_model = WhisperForConditionalGeneration.from_pretrained(whisper_base_model).to(device)

whisper_model.generation_config.forced_decoder_ids = None
whisper_model.generation_config.suppress_tokens = []
whisper_model.config.use_cache = False

class WhisperDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sr = torchaudio.load(row["audio_path"])
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        audio_np = waveform.squeeze().numpy()

        input_features = self.processor.feature_extractor(audio_np, sampling_rate=16000).input_features[0]
        labels = self.processor.tokenizer(str(row["sentence"])).input_ids
        return {"input_features": input_features, "labels": labels}

whisper_train_set = WhisperDataset(train_df, whisper_processor)
whisper_eval_set = WhisperDataset(eval_df, whisper_processor)

@dataclasses.dataclass
class DataCollatorWhisper:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_feats = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_feats, return_tensors="pt")
        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

whisper_collator = DataCollatorWhisper(processor=whisper_processor)

def compute_whisper_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = whisper_processor.tokenizer.pad_token_id
    pred_str = whisper_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = whisper_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": compute_wer(pred_str, label_str), "cer": compute_cer(pred_str, label_str)}

whisper_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper_checkpoints",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=1,
    dataloader_num_workers=2,
)

whisper_trainer = Seq2SeqTrainer(
    args=whisper_args,
    model=whisper_model,
    train_dataset=whisper_train_set,
    eval_dataset=whisper_eval_set,
    data_collator=whisper_collator,
    compute_metrics=compute_whisper_metrics,
    processing_class=whisper_processor.feature_extractor,
)

# Resume from checkpoint if present
last_ckpt = None
if os.path.isdir(whisper_args.output_dir):
    ckpts = glob.glob(os.path.join(whisper_args.output_dir, "checkpoint-*"))
    if ckpts:
        last_ckpt = max(ckpts, key=lambda p: int(p.rsplit("-", 1)[-1]))

print(f"[*] Training Whisper-Small (Resume: {last_ckpt})...")
whisper_trainer.train(resume_from_checkpoint=last_ckpt)

save_whisper_path = "/kaggle/working/bangla_asr_best"
whisper_trainer.save_model(save_whisper_path)
whisper_processor.save_pretrained(save_whisper_path)
print(f"[✓] Whisper Checkpoint Saved → {save_whisper_path}")

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

[*] Training Whisper-Small (Resume: None)...


Step,Training Loss,Validation Loss,Wer,Cer
100,1.357390,0.320277,53.661263,70.385914


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ==============================================================================
# CELL 3: Model 2 — Meta MMS-300M Fine-Tuning (Stable 300M CTC Model)
# ==============================================================================
# Purge Whisper VRAM allocation
del whisper_model
del whisper_trainer
gc.collect()
torch.cuda.empty_cache()

from transformers import Wav2Vec2ForCTC, AutoProcessor, TrainingArguments, Trainer

mms_model_id = "facebook/mms-300m"
print(f"[*] Initializing Meta MMS-300M on {device}...")

mms_processor = AutoProcessor.from_pretrained(mms_model_id)
mms_processor.tokenizer.set_target_lang("ben")

mms_model = Wav2Vec2ForCTC.from_pretrained(
    mms_model_id,
    target_lang="ben",
    ignore_mismatched_sizes=True,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.05,
    ctc_loss_reduction="mean",
    pad_token_id=mms_processor.tokenizer.pad_token_id,
    vocab_size=len(mms_processor.tokenizer.get_vocab()),
).to(device)

mms_model.freeze_feature_encoder()
mms_model.gradient_checkpointing_enable()

class MMSDataset(Dataset):
    def __init__(self, df, processor, max_samples=160000):
        self.df = df
        self.processor = processor
        self.max_samples = max_samples

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sr = torchaudio.load(row["audio_path"])
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        
        audio_np = waveform.squeeze().numpy()
        if len(audio_np) > self.max_samples:
            audio_np = audio_np[:self.max_samples]
            
        input_values = self.processor(audio_np, sampling_rate=16000).input_values[0]
        labels = self.processor.tokenizer(str(row["sentence"])).input_ids
        return {"input_values": input_values, "labels": labels}

mms_train_set = MMSDataset(train_df, mms_processor)
mms_eval_set = MMSDataset(eval_df, mms_processor)

@dataclasses.dataclass
class DataCollatorCTC:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_values = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(input_values, padding=True, return_tensors="pt")
        labels = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(labels, padding=True, return_tensors="pt")
        masked_labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = masked_labels
        return batch

mms_collator = DataCollatorCTC(processor=mms_processor)

def compute_mms_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = mms_processor.tokenizer.pad_token_id
    pred_str = mms_processor.batch_decode(pred_ids)
    label_str = mms_processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": compute_wer(pred_str, label_str), "cer": compute_cer(pred_str, label_str)}

mms_args = TrainingArguments(
    output_dir="/kaggle/working/mms_checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    max_steps=300,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    per_device_eval_batch_size=4,
    eval_steps=75,
    save_steps=75,
    logging_steps=25,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=1,
    dataloader_num_workers=2,
)

mms_trainer = Trainer(
    model=mms_model,
    args=mms_args,
    train_dataset=mms_train_set,
    eval_dataset=mms_eval_set,
    data_collator=mms_collator,
    compute_metrics=compute_mms_metrics,
    processing_class=mms_processor.feature_extractor,
)

print("[*] Commencing Meta MMS-300M Fine-Tuning...")
mms_trainer.train()

save_mms_path = "/kaggle/working/mms_asr_best"
mms_model.save_pretrained(save_mms_path)
mms_processor.save_pretrained(save_mms_path)
print(f"[✓] Meta MMS-300M Saved → {save_mms_path}")

In [ ]:
# ==============================================================================
# CELL 4: Comparative Evaluation: Whisper vs. MMS across 20 Districts
# ==============================================================================
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoProcessor, Wav2Vec2ForCTC, WhisperProcessor, WhisperForConditionalGeneration

# Load Whisper
whisper_load_path = "/kaggle/working/bangla_asr_best" if os.path.exists("/kaggle/working/bangla_asr_best") else "bangla-speech-processing/BanglaASR"
whisper_proc = WhisperProcessor.from_pretrained("bangla-speech-processing/BanglaASR")
whisper_model = WhisperForConditionalGeneration.from_pretrained(whisper_load_path).to(device)
whisper_model.eval()

# Load MMS-300M
mms_load_path = "/kaggle/working/mms_asr_best" if os.path.exists("/kaggle/working/mms_asr_best") else "facebook/mms-300m"
mms_saved_proc = AutoProcessor.from_pretrained(mms_load_path)
mms_saved_proc.tokenizer.set_target_lang("ben")
mms_saved_model = Wav2Vec2ForCTC.from_pretrained(mms_load_path).to(device)
mms_saved_model.eval()

eval_results_list = []
print("[*] Running Comparative Validation Inference...")

for idx in tqdm(range(len(eval_df)), desc="Dual-Model Inference"):
    row = eval_df.iloc[idx]
    waveform, sr = torchaudio.load(row["audio_path"])
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    audio_np = waveform.squeeze().numpy()

    # Whisper
    whisper_feats = whisper_proc.feature_extractor(audio_np, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        w_pred_ids = whisper_model.generate(input_features=whisper_feats)
        whisper_transcription = whisper_proc.decode(w_pred_ids[0], skip_special_tokens=True)

    # MMS
    mms_inputs = mms_saved_proc(audio_np, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        m_logits = mms_saved_model(mms_inputs).logits
        m_pred_ids = torch.argmax(m_logits, dim=-1)
        mms_transcription = mms_saved_proc.decode(m_pred_ids[0])

    eval_results_list.append({
        "district": row["district_token"],
        "district_bn": row["জেলা"],
        "division_bn": row["বিভাগ"],
        "reference": str(row["sentence"]),
        "whisper_pred": whisper_transcription,
        "mms_pred": mms_transcription
    })

comparison_df = pd.DataFrame(eval_results_list)

# Aggregate District Metrics
district_metrics = []
for dist, grp in comparison_df.groupby("district"):
    refs = grp["reference"].tolist()
    w_preds = grp["whisper_pred"].tolist()
    m_preds = grp["mms_pred"].tolist()
    
    district_metrics.append({
        "District": dist.capitalize(),
        "জেলা": grp["district_bn"].iloc[0],
        "বিভাগ": grp["division_bn"].iloc[0],
        "Samples": len(grp),
        "Whisper WER (%)": round(compute_wer(w_preds, refs), 2),
        "Whisper CER (%)": round(compute_cer(w_preds, refs), 2),
        "MMS WER (%)": round(compute_wer(m_preds, refs), 2),
        "MMS CER (%)": round(compute_cer(m_preds, refs), 2),
    })

benchmark_summary_df = pd.DataFrame(district_metrics).sort_values("Whisper WER (%)").reset_index(drop=True)
benchmark_summary_df.to_csv("/kaggle/working/dual_model_district_benchmark.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*85)
print("             20-DISTRICT ASR BENCHMARK (WHISPER vs. META MMS-300M)")
print("="*85)
print(benchmark_summary_df.to_string(index=False))

In [ ]:
# ==============================================================================
# CELL 5: Cascaded Normalization Pipeline & Output File Exports
# ==============================================================================
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

nlg_model_path = "csebuetnlp/banglat5"
print(f"[*] Initializing Dialect Normalizer: {nlg_model_path}...")

nlg_tokenizer = AutoTokenizer.from_pretrained(nlg_model_path)
nlg_model = AutoModelForSeq2SeqLM.from_pretrained(nlg_model_path).to(device)
nlg_model.eval()

def normalize_dialect_to_standard(dialect_text: str) -> str:
    input_text = f"translate Bengali dialect to Standard: {dialect_text}"
    inputs = nlg_tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True).to(device)
    with torch.no_grad():
        outputs = nlg_model.generate(**inputs, max_length=128, num_beams=4)
    return nlg_tokenizer.decode(outputs[0], skip_special_tokens=True)

e2e_samples = []
for idx in range(min(15, len(comparison_df))):
    row = comparison_df.iloc[idx]
    std_prediction = normalize_dialect_to_standard(row["whisper_pred"])
    e2e_samples.append({
        "District": row["district_bn"],
        "Division": row["division_bn"],
        "Regional Audio Transcript (Ground Truth)": row["reference"],
        "ASR Recognized Dialect": row["whisper_pred"],
        "Normalized Standard Bangla Output": std_prediction
    })

e2e_df = pd.DataFrame(e2e_samples)
e2e_df.to_csv("/kaggle/working/e2e_speech_to_standard_results.csv", index=False, encoding="utf-8-sig")

# Export LaTeX Table
latex_table = benchmark_summary_df[[
    "District", "Samples", "Whisper WER (%)", "Whisper CER (%)", "MMS WER (%)", "MMS CER (%)"
]].to_latex(
    index=False,
    caption="Comparative Performance of Whisper vs. Meta MMS across 20 Dialectal Districts of Bangladesh.",
    label="tab:asr_benchmark"
)

with open("/kaggle/working/table_asr_benchmark.tex", "w") as f:
    f.write(latex_table)

print("[✓] Saved E2E Pipeline Results and LaTeX table to /kaggle/working/")

In [ ]:
# ==============================================================================
# CELL 6: Publication-Grade Visualizations (.pdf & .png)
# ==============================================================================
import matplotlib
import matplotlib.pyplot as plt

OUTPUT_DIR = "/kaggle/working/paper_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
matplotlib.rcParams["axes.edgecolor"] = "#334155"
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["grid.color"] = "#E2E8F0"
matplotlib.rcParams["grid.linestyle"] = "--"
matplotlib.rcParams["grid.alpha"] = 0.7

PALETTE = {"whisper": "#1E40AF", "mms": "#0D9488", "card_bg": "#F8FAFC", "bg": "#FFFFFF"}

# ── FIG 1: 20-District Comparative Bar Chart (WER & CER) ──────────────────────
df_sorted = benchmark_summary_df.sort_values("Whisper WER (%)", ascending=True).reset_index(drop=True)
y_pos = np.arange(len(df_sorted))
bar_h = 0.38

fig, (ax_wer, ax_cer) = plt.subplots(1, 2, figsize=(14, 7.5), dpi=300, facecolor=PALETTE["bg"])
for ax in [ax_wer, ax_cer]:
    ax.set_facecolor(PALETTE["card_bg"])

ax_wer.barh(y_pos - bar_h/2, df_sorted["Whisper WER (%)"], height=bar_h, color=PALETTE["whisper"], label="Whisper-Small")
ax_wer.barh(y_pos + bar_h/2, df_sorted["MMS WER (%)"], height=bar_h, color=PALETTE["mms"], label="Meta MMS-300M")
ax_wer.set_yticks(y_pos)
ax_wer.set_yticklabels(df_sorted["District"], fontsize=9)
ax_wer.set_xlabel("Word Error Rate (WER %)", fontsize=10)
ax_wer.set_title("(a) Word Error Rate by District", fontsize=11, fontweight="bold")
ax_wer.invert_yaxis()
ax_wer.legend(loc="lower right")
ax_wer.grid(axis="x")

ax_cer.barh(y_pos - bar_h/2, df_sorted["Whisper CER (%)"], height=bar_h, color=PALETTE["whisper"], label="Whisper-Small")
ax_cer.barh(y_pos + bar_h/2, df_sorted["MMS CER (%)"], height=bar_h, color=PALETTE["mms"], label="Meta MMS-300M")
ax_cer.set_yticks(y_pos)
ax_cer.set_yticklabels([])
ax_cer.set_xlabel("Character Error Rate (CER %)", fontsize=10)
ax_cer.set_title("(b) Character Error Rate by District", fontsize=11, fontweight="bold")
ax_cer.invert_yaxis()
ax_cer.legend(loc="lower right")
ax_cer.grid(axis="x")

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_district_asr_benchmark.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_district_asr_benchmark.png"), dpi=300, bbox_inches="tight")
plt.close()

# ── FIG 2: Division-Level Aggregated Bar Chart ────────────────────────────────
div_map = {
    "Barisal": "Barisal", "Bhola": "Barisal", "Jhalokati": "Barisal", "Patuakhali": "Barisal",
    "Chittagong": "Chittagong", "Comilla": "Chittagong", "Feni": "Chittagong", "Brahmanbaria": "Chittagong", "Noakhali": "Chittagong", "Lakshmipur": "Chittagong",
    "Dhaka": "Dhaka", "Kishoreganj": "Dhaka", "Narsingdi": "Dhaka", "Tangail": "Dhaka", "Mymensingh": "Mymensingh",
    "Khulna": "Khulna", "Jessore": "Khulna", "Kushtia": "Khulna", "Jhenaidah": "Khulna",
    "Rajshahi": "Rajshahi", "Bogura": "Rajshahi", "Natore": "Rajshahi", "Pabna": "Rajshahi",
    "Rangpur": "Rangpur", "Sylhet": "Sylhet"
}
df_sorted["Division"] = df_sorted["District"].map(div_map).fillna("Chittagong")
div_summary = df_sorted.groupby("Division").agg({"Whisper WER (%)": "mean", "MMS WER (%)": "mean"}).sort_values("Whisper WER (%)").reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=300, facecolor=PALETTE["bg"])
ax.set_facecolor(PALETTE["card_bg"])
x_idx = np.arange(len(div_summary))
ax.bar(x_idx - 0.18, div_summary["Whisper WER (%)"], width=0.35, label="Whisper-Small", color=PALETTE["whisper"])
ax.bar(x_idx + 0.18, div_summary["MMS WER (%)"], width=0.35, label="Meta MMS-300M", color=PALETTE["mms"])
ax.set_xticks(x_idx)
ax.set_xticklabels(div_summary["Division"], fontsize=10)
ax.set_ylabel("Mean WER (%)", fontsize=10)
ax.set_title("Aggregated ASR Performance across Administrative Divisions of Bangladesh", fontsize=11, fontweight="bold")
ax.legend()
ax.grid(axis="y")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_division_level_aggregation.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_division_level_aggregation.png"), dpi=300, bbox_inches="tight")
plt.close()

print(f"\n[✓] All publication figures and CSV logs generated in: {OUTPUT_DIR}")